# Blitz ASR - Quickstart Guide

Get Blitz running on a g6.xlarge instance (NVIDIA L4 GPU) in under 15 minutes.

## Target Environment
- **Instance**: g6.xlarge (NVIDIA L4 24GB VRAM)
- **OS**: Amazon Linux 2023 or Ubuntu 22.04
- **Docker**: With NVIDIA Container Toolkit

## Workflow Options
1. **Build from Source** - Full control, first run ~15-20 min
2. **Pull from ECR** - Faster if image already exists

---
## 1. Prerequisites Check

Verify your environment is ready for GPU inference.

In [ ]:
# Check NVIDIA GPU is available
!nvidia-smi

In [ ]:
# Check Docker with GPU support
!docker run --rm --gpus all nvidia/cuda:12.0-base nvidia-smi

In [ ]:
# Check Docker Compose version (v2 required)
!docker compose version

---
## 2A. Option A: Build from Source

Build the image locally. First run downloads models (~15-20 min).

In [ ]:
# Navigate to project root
import os
os.chdir('..')  # Adjust if needed
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Create .env from example (if not exists)
!cp -n .env.example .env 2>/dev/null || echo ".env already exists"
!cat .env | head -20

In [ ]:
# Build and start (first run: ~15-20 min for model downloads)
!docker compose up -d --build

In [ ]:
# Watch logs until "Application startup complete"
# Press Ctrl+C when ready
!docker compose logs -f

---
## 2B. Option B: Pull from ECR

Use a pre-built image from Amazon ECR (faster for subsequent deployments).

In [ ]:
# Configuration - UPDATE THESE VALUES
ACCOUNT_ID = "123456789012"  # Your AWS account ID
REGION = "us-east-1"         # Your AWS region
IMAGE_TAG = "latest"         # Image tag

ECR_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/blitz-asr:{IMAGE_TAG}"
print(f"ECR URI: {ECR_URI}")

In [ ]:
# Authenticate with ECR
!aws ecr get-login-password --region {REGION} | \
  docker login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com

In [ ]:
# Pull the image
!docker pull {ECR_URI}

In [ ]:
# Run the container
!docker run --gpus all -d -p 8000:8000 --name blitz {ECR_URI}

In [ ]:
# Watch logs until models are loaded
!docker logs -f blitz

---
## 3. Health Check

Verify the service is running and models are loaded.

In [ ]:
import requests
import time

# Wait for service to be ready
max_retries = 30
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:8000/health", timeout=5)
        if response.status_code == 200:
            health = response.json()
            print(f"Status: {health.get('status', 'unknown')}")
            print(f"Models loaded: {health.get('models', {})}")
            break
    except requests.exceptions.ConnectionError:
        print(f"Waiting for service... ({i+1}/{max_retries})")
        time.sleep(5)
else:
    print("Service did not start. Check logs with: docker logs blitz")

---
## 4. Single File Transcription

Test with an English audio file.

In [ ]:
import requests

# Transcribe English test file
with open("tests/fixtures/english.mp3", "rb") as f:
    response = requests.post(
        "http://localhost:8000/transcribe",
        files={"file": ("english.mp3", f, "audio/mpeg")},
        data={"format": "json"}
    )

result = response.json()
print(f"Language: {result['language']}")
print(f"Model tier: {result['model']}")
print(f"Audio duration: {result['metrics']['audio_duration_sec']:.1f}s")
print(f"RTF: {result['metrics']['real_time_factor']:.4f}")
print(f"\nTranscript (first 500 chars):\n{result['transcription'][:500]}...")

---
## 5. Multi-Language Testing

Test with multiple EU language samples to verify Tier 2 routing.

In [ ]:
from pathlib import Path
from tqdm import tqdm
import time
import requests

fixtures = Path("tests/fixtures")

# Languages to test (must have corresponding .mp3 files)
languages = [
    "english", "french", "german", "spanish", "italian",
    "portuguese", "dutch", "polish", "czech", "russian"
]

results = []
for lang in tqdm(languages, desc="Testing languages"):
    filepath = fixtures / f"{lang}.mp3"
    if not filepath.exists():
        print(f"  Skipping {lang} (file not found)")
        continue
    
    start = time.time()
    with open(filepath, "rb") as f:
        resp = requests.post(
            "http://localhost:8000/transcribe",
            files={"file": (filepath.name, f, "audio/mpeg")},
            data={"format": "json"}
        )
    elapsed = time.time() - start
    
    if resp.status_code == 200:
        data = resp.json()
        results.append({
            "file": lang,
            "detected_lang": data["language"],
            "model": data["model"],
            "duration_sec": data["metrics"]["audio_duration_sec"],
            "processing_sec": elapsed,
            "rtf": data["metrics"]["real_time_factor"]
        })
    else:
        print(f"  Error for {lang}: {resp.status_code}")

In [ ]:
# Display results
import polars as pl

df = pl.DataFrame(results)
print(df)

print(f"\nSummary:")
print(f"  Total audio: {df['duration_sec'].sum():.1f}s")
print(f"  Total processing: {df['processing_sec'].sum():.1f}s")
print(f"  Average RTF: {df['rtf'].mean():.4f}")
print(f"  Average RTFx: {1/df['rtf'].mean():.0f}x real-time")

---
## 6. Performance Benchmarking

Measure throughput with larger test batches.

In [ ]:
# Find all audio files in fixtures
fixtures = Path("tests/fixtures")
audio_files = list(fixtures.glob("*.mp3")) + list(fixtures.glob("*.wav"))

# Filter out very large files for quick benchmarking
audio_files = [f for f in audio_files if f.stat().st_size < 10_000_000]  # <10MB

print(f"Found {len(audio_files)} audio files for benchmarking")
for f in audio_files[:10]:
    print(f"  {f.name}: {f.stat().st_size / 1024:.0f} KB")

In [ ]:
from tqdm import tqdm
import time
import requests

benchmark_results = []
total_start = time.time()

for filepath in tqdm(audio_files, desc="Benchmarking"):
    start = time.time()
    with open(filepath, "rb") as f:
        resp = requests.post(
            "http://localhost:8000/transcribe",
            files={"file": (filepath.name, f)},
            data={"format": "json"}
        )
    elapsed = time.time() - start
    
    if resp.status_code == 200:
        data = resp.json()
        benchmark_results.append({
            "file": filepath.name,
            "language": data["language"],
            "model": data["model"],
            "duration_sec": data["metrics"]["audio_duration_sec"],
            "processing_sec": elapsed,
            "rtf": data["metrics"]["real_time_factor"]
        })

total_elapsed = time.time() - total_start

In [ ]:
import polars as pl

df = pl.DataFrame(benchmark_results)
print(df)

total_audio = df["duration_sec"].sum()
total_proc = df["processing_sec"].sum()

print(f"\n" + "="*60)
print(f"BENCHMARK SUMMARY")
print(f"="*60)
print(f"Files processed: {len(benchmark_results)}")
print(f"Total audio: {total_audio:.1f}s ({total_audio/60:.1f} min)")
print(f"Total wall time: {total_elapsed:.1f}s")
print(f"Throughput: {total_audio/total_elapsed:.2f} audio-sec/wall-sec")
print(f"Average RTF: {df['rtf'].mean():.4f}")
print(f"Average RTFx: {1/df['rtf'].mean():.0f}x real-time")
print(f"")
print(f"By model tier:")
for tier in df["model"].unique().to_list():
    tier_df = df.filter(pl.col("model") == tier)
    print(f"  {tier}: {len(tier_df)} files, avg RTFx: {1/tier_df['rtf'].mean():.0f}x")

---
## 6A. Load Testing

For more rigorous load testing with concurrent requests, use the provided test scripts.

In [ ]:
# Concurrent load test with 40 parallel requests
# Uses files from tests/fixtures/ by default
!python3 tests/test_load.py -n 40

In [ ]:
# Continuous load test for specified duration
# Useful for measuring sustained throughput and memory stability
!python3 tests/test_load_loop.py --minutes 1 --test-dir tests/fixtures/mono-wav-tests/

In [ ]:
# Quick curl test with JSON output
!curl -X POST "http://localhost:8000/transcribe" \
    -H "accept: application/json" \
    -H "Content-Type: multipart/form-data" \
    -F "file=@tests/fixtures/french.mp3" | jq

In [ ]:
# Curl with jq filtering to extract specific fields
!curl -s -X POST "http://localhost:8000/transcribe" \
    -F "file=@tests/fixtures/english.mp3" \
    -F "format=json" | jq '{
      language: .language,
      model: .model,
      audio_duration: .metrics.audio_duration_sec,
      rtf: .metrics.real_time_factor,
      text_preview: .transcription[0:200]
    }'

In [ ]:
# Curl test with SRT subtitle format
!curl -s -X POST "http://localhost:8000/transcribe" \
    -F "file=@tests/fixtures/german.mp3" \
    -F "format=srt"

---
## 7. Output Formats

Blitz supports multiple output formats: JSON, SRT, VTT, and plain text.

In [ ]:
# JSON format (default)
with open("tests/fixtures/english.mp3", "rb") as f:
    resp = requests.post(
        "http://localhost:8000/transcribe",
        files={"file": f},
        data={"format": "json"}
    )
print("JSON Format:")
print(resp.json())

In [ ]:
# SRT format (subtitles)
with open("tests/fixtures/english.mp3", "rb") as f:
    resp = requests.post(
        "http://localhost:8000/transcribe",
        files={"file": f},
        data={"format": "srt"}
    )
print("SRT Format:")
print(resp.text[:500])

In [ ]:
# VTT format (web subtitles)
with open("tests/fixtures/english.mp3", "rb") as f:
    resp = requests.post(
        "http://localhost:8000/transcribe",
        files={"file": f},
        data={"format": "vtt"}
    )
print("VTT Format:")
print(resp.text[:500])

In [ ]:
# Plain text format
with open("tests/fixtures/english.mp3", "rb") as f:
    resp = requests.post(
        "http://localhost:8000/transcribe",
        files={"file": f},
        data={"format": "text"}
    )
print("Text Format:")
print(resp.text[:500])

---
## 8. Push to ECR

Save your built image to ECR for reuse across instances.

In [ ]:
# Configuration - UPDATE THESE VALUES
ACCOUNT_ID = "123456789012"  # Your AWS account ID
REGION = "us-east-1"         # Your AWS region
IMAGE_TAG = "latest"         # Image tag

In [ ]:
# Push to ECR using the provided script
!./scripts/push_to_ecr.sh {ACCOUNT_ID} {REGION} {IMAGE_TAG}

---
## 9. Cleanup

Stop and remove the container when done.

In [ ]:
# If using docker compose
!docker compose down

In [ ]:
# If using docker run directly
!docker stop blitz && docker rm blitz

---
## Next Steps

- **Production Scale**: See `02_aws_batch_deployment.ipynb` for S3-triggered batch processing
- **API Integration**: See `docs/API_DOCUMENTATION.md` for full API reference
- **Architecture**: See `docs/ARCHITECTURE.md` for system design details
- **Benchmarks**: See `docs/benchmark-results.md` for detailed performance data